In [1]:
# ═══════════════════════════════════════════════════════════════════
# PATCH CELL — Re-run BLS with correct SNR, recompute all affected
# columns (labels, engineered features, global/local views), and
# overwrite the saved files.
#
# Assumes these are already in scope:
#   superdf        — raw parquet DataFrame (target_id, time, flux, flux_err)
#   df             — built dataset DataFrame (from pickle)
#   global_views   — (N, 2000) float32 numpy array
#   local_views    — (N, 201)  float32 numpy array
#   scalar_feats   — (N, n_cols) float32 numpy array
#
# Files overwritten:
#   final_sector1_processed/final_tess_dataset.pkl
#   final_sector1_processed/global_views.npy
#   final_sector1_processed/local_views.npy
#   final_sector1_processed/labels.npy
#   final_sector1_processed/scalar_features.npy
# ═══════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import pickle, warnings
from scipy.signal import savgol_filter
from scipy.stats import median_abs_deviation, skew, kurtosis
from astropy.timeseries import BoxLeastSquares
import astropy.units as u

warnings.filterwarnings("ignore")

with open('final_sector1_processed/final_tess_dataset.pkl', 'rb') as f:
    df = pickle.load(f)

global_views  = np.load('final_sector1_processed/global_views.npy')
local_views   = np.load('final_sector1_processed/local_views.npy')
scalar_feats  = np.load('final_sector1_processed/scalar_features.npy')

superdf=pd.read_parquet('raw_data/science_sector1.parquet')

# ── Config (must match what the builder used) ────────────────────
OUT_DIR        = "final_sector1_processed"
PERIOD_MIN     = 0.5
PERIOD_MAX     = 13.0
N_PERIODS      = 5000
DURATION_GRID  = np.geomspace(0.02, PERIOD_MIN * 0.85, 20)
SIGMA_CLIP     = 4.0
SG_WINDOW      = 721
SG_POLY        = 3
GLOBAL_BINS    = 2000
LOCAL_BINS     = 201
LOCAL_DUR_MUL  = 3.0
SNR_CONFIRMED  = 15.0
SNR_CANDIDATE  = 7.0
N_AUGMENTS     = 2                # must match builder so index math works
PHASE_JITTER_STD = 0.01
FLUX_NOISE_STD   = 0.002

SCALAR_COLS = [
    "bls_period", "bls_depth", "bls_duration", "bls_snr",
    "transit_depth", "depth_snr", "ingress_egress_asymmetry",
    "flat_bottom_ratio", "oot_scatter", "transit_skewness",
    "secondary_depth", "secondary_primary_ratio",
    "odd_even_depth_diff", "odd_even_depth_ratio",
    "baseline_kurtosis", "n_transits", "duty_cycle",
    "duration_over_period", "median_flux_err", "mean_flux_err",
    "n_cadences", "lc_span_days", "duty_cycle_obs",
]

# ── Helper functions (copied from builder, no changes) ───────────

def _sigma_clip(flux):
    med = np.nanmedian(flux)
    mad = median_abs_deviation(flux, nan_policy="omit")
    if mad == 0:
        return np.ones(len(flux), dtype=bool)
    return np.abs(flux - med) < SIGMA_CLIP * 1.4826 * mad

def _detrend(flux):
    n = len(flux)
    w = min(SG_WINDOW, n if n % 2 == 1 else n - 1)
    if w <= SG_POLY:
        return flux
    baseline = savgol_filter(flux, window_length=w, polyorder=SG_POLY)
    baseline = np.where(np.abs(baseline) < 1e-9, 1.0, baseline)
    return flux / baseline

def _median_bin(phase, flux, n_bins, pmin, pmax):
    edges  = np.linspace(pmin, pmax, n_bins + 1)
    result = np.full(n_bins, np.nan, dtype=np.float32)
    for i in range(n_bins):
        m = (phase >= edges[i]) & (phase < edges[i + 1])
        if m.any():
            result[i] = np.nanmedian(flux[m])
    nans = np.isnan(result)
    if nans.any() and not nans.all():
        idx = np.arange(n_bins)
        result[nans] = np.interp(idx[nans], idx[~nans], result[~nans])
    elif nans.all():
        result[:] = 0.0
    return result

def _robust_norm(view, transit_mask):
    base = view[~transit_mask]
    if len(base) == 0 or np.all(np.isnan(base)):
        base = view
    med = np.nanmedian(base)
    mad = median_abs_deviation(base, nan_policy="omit") * 1.4826
    if mad < 1e-9:
        mad = float(np.nanstd(base)) or 1.0
    return ((view - med) / mad).astype(np.float32)

def _pad(arr, length):
    if len(arr) >= length:
        return arr[:length]
    return np.concatenate([arr, np.zeros(length - len(arr), dtype=arr.dtype)])

def _run_bls(time, flux):
    model   = BoxLeastSquares(time * u.day, flux)
    periods = np.geomspace(PERIOD_MIN, PERIOD_MAX, N_PERIODS)
    result  = model.power(periods * u.day, duration=DURATION_GRID * u.day,
                          method="fast", objective="snr")
    best     = int(np.argmax(result.power))
    period   = float(result.period[best].value)
    t0       = float(result.transit_time[best].value)
    depth    = float(abs(result.depth[best]))
    duration = float(result.duration[best].value)
    in_mask  = model.transit_mask(time * u.day, result.period[best],
                                   result.duration[best], result.transit_time[best])
    in_f, out_f = flux[in_mask], flux[~in_mask]
    if len(in_f) < 3 or len(out_f) < 10:
        snr = 0.0
    else:
        d   = abs(float(np.median(out_f) - np.median(in_f)))
        std = float(np.std(out_f))
        snr = d / (std / np.sqrt(len(in_f))) if std > 0 else 0.0
    return period, t0, depth, duration, snr

def _eng_features(time, flux, period, t0, duration):
    feats = {}
    phase       = ((time - t0) / period + 0.5) % 1.0 - 0.5
    half_dur_ph = (duration / period) / 2.0
    in_t  = np.abs(phase) < half_dur_ph
    out_t = ~in_t
    if in_t.sum() > 5 and out_t.sum() > 10:
        tf, bf   = flux[in_t], flux[out_t]
        b_med    = float(np.nanmedian(bf))
        b_std    = float(np.nanstd(bf))
        feats["transit_depth"] = float(b_med - np.nanmedian(tf))
        feats["depth_snr"]     = feats["transit_depth"] / b_std if b_std > 0 else 0.0
        left  = flux[(phase > -half_dur_ph) & (phase < 0)]
        right = flux[(phase > 0)            & (phase < half_dur_ph)]
        feats["ingress_egress_asymmetry"] = (
            float(np.nanmedian(left) - np.nanmedian(right))
            if len(left) > 2 and len(right) > 2 else 0.0)
        flat_bins = tf < (b_med - feats["transit_depth"] * 0.9)
        feats["flat_bottom_ratio"] = float(flat_bins.sum() / max(len(tf), 1))
        feats["oot_scatter"]       = b_std
        feats["transit_skewness"]  = float(skew(tf))
    else:
        for k in ["transit_depth","depth_snr","ingress_egress_asymmetry",
                  "flat_bottom_ratio","oot_scatter","transit_skewness"]:
            feats[k] = np.nan
    sec_mask = np.abs(phase - 0.5) < half_dur_ph
    if sec_mask.sum() > 3 and out_t.sum() > 10:
        b_med = float(np.nanmedian(flux[out_t]))
        feats["secondary_depth"] = float(b_med - np.nanmedian(flux[sec_mask]))
        pd_ = feats.get("transit_depth", np.nan)
        feats["secondary_primary_ratio"] = (float(feats["secondary_depth"] / pd_)
                                             if np.isfinite(pd_) and pd_ > 0 else np.nan)
    else:
        feats["secondary_depth"] = feats["secondary_primary_ratio"] = np.nan
    t_num = np.floor((time - t0) / period).astype(int)
    ph2   = ((time - t0) / (period / 2.0) + 0.5) % 1.0 - 0.5
    o_m   = (np.abs(ph2) < half_dur_ph) & (t_num % 2 == 0)
    e_m   = (np.abs(ph2) < half_dur_ph) & (t_num % 2 == 1)
    if o_m.sum() > 3 and e_m.sum() > 3 and out_t.sum() > 10:
        b_med = float(np.nanmedian(flux[out_t]))
        od, ed = float(b_med - np.nanmedian(flux[o_m])), float(b_med - np.nanmedian(flux[e_m]))
        md = abs(od + ed) / 2.0
        feats["odd_even_depth_diff"]  = float(abs(od - ed))
        feats["odd_even_depth_ratio"] = float(abs(od - ed) / max(md, 1e-9))
    else:
        feats["odd_even_depth_diff"] = feats["odd_even_depth_ratio"] = np.nan
    feats["baseline_kurtosis"] = float(kurtosis(flux[out_t])) if out_t.sum() > 10 else np.nan
    span = float(np.nanmax(time) - np.nanmin(time))
    feats["n_transits"]           = float(span / period) if period > 0 else np.nan
    feats["duty_cycle"]           = float(duration / period) if period > 0 else np.nan
    feats["duration_over_period"] = feats["duty_cycle"]
    return feats

def _make_views(time, flux, period, t0, duration):
    phase = ((time - t0) / period + 0.5) % 1.0 - 0.5
    order = np.argsort(phase)
    phase, flux = phase[order], flux[order]
    graw  = _median_bin(phase, flux, GLOBAL_BINS, -0.5, 0.5)
    bc    = (np.arange(GLOBAL_BINS) + 0.5) / GLOBAL_BINS - 0.5
    hdp   = max(duration, 0.02) / period
    gview = _robust_norm(graw, np.abs(bc) < hdp)
    hw    = np.clip(LOCAL_DUR_MUL * hdp, 0.02, 0.48)
    lm    = np.abs(phase) <= hw
    lp, lf = phase[lm], flux[lm]
    if lp.size < 10:
        lview = np.zeros(LOCAL_BINS, dtype=np.float32)
    else:
        lraw  = _median_bin(lp, lf, LOCAL_BINS, -hw, hw)
        ne    = max(1, LOCAL_BINS // 10)
        emask = np.zeros(LOCAL_BINS, dtype=bool)
        emask[:ne] = emask[-ne:] = True
        lview = _robust_norm(lraw, ~emask)
    return _pad(gview, GLOBAL_BINS), _pad(lview, LOCAL_BINS)

# ── Main patch loop ──────────────────────────────────────────────

# Build a lookup from target_id → preprocessed (time, flux) from superdf
# so we don't re-preprocess the same star for each augmented copy.
print("Pre-processing raw light curves from superdf...")
raw_lookup = {}   # target_id (int) → (time, flux) after clip+detrend+norm
for _, row in superdf.iterrows():
    tic  = int(row["target_id"])
    time = np.asarray(row["time"], dtype=np.float64)
    flux = np.asarray(row["flux"], dtype=np.float64)
    good = _sigma_clip(flux)
    time, flux = time[good], flux[good]
    if len(time) < 200:
        continue
    flux = _detrend(flux)
    med  = float(np.nanmedian(flux))
    if abs(med) < 1e-9:
        continue
    flux = flux / med
    raw_lookup[tic] = (time, flux)

print(f"  {len(raw_lookup)} stars preprocessed.\n")

# Identify the real (non-augmented) rows and their order
# Each real row was inserted first; augmented copies follow it.
# We iterate real rows, compute new BLS + features + views,
# then propagate views to the augmented copies with the same noise logic.

real_mask = df["augmented"].isna().values
real_idx  = np.where(real_mask)[0]
print(f"Found {len(real_idx)} real rows and {(~real_mask).sum()} augmented rows.")

rng = np.random.default_rng(42)             # same seed as builder for reproducibility

# Collect updated columns
new_records = {}   # row_index → dict of updated scalar columns
new_gviews  = global_views.copy()
new_lviews  = local_views.copy()

print(f"Running corrected BLS on {len(real_idx)} real records...")
skipped = 0

for pos, ri in enumerate(real_idx):
    tic = int(df.at[ri, "target_id"])

    if tic not in raw_lookup:
        skipped += 1
        if (pos + 1) % 100 == 0:
            print(f"  [{pos+1}/{len(real_idx)}] {skipped} skipped so far")
        continue

    time, flux = raw_lookup[tic]

    # BLS
    try:
        period, t0, bls_depth, bls_dur, bls_snr = _run_bls(time, flux)
    except Exception as e:
        print(f"  [BLS FAIL] TIC {tic}: {e}")
        skipped += 1
        continue

    if not np.isfinite(period) or not np.isfinite(t0):
        skipped += 1
        continue

    duration = bls_dur if np.isfinite(bls_dur) else 0.1
    label    = 0 if bls_snr >= SNR_CONFIRMED else (1 if bls_snr >= SNR_CANDIDATE else 2)

    # Engineered features
    eng = _eng_features(time, flux, period, t0, duration)

    # Views
    gv, lv = _make_views(time, flux, period, t0, duration)

    # Store scalar updates for this real row
    new_records[ri] = {
        "label":        label,
        "fold_period":  float(period),
        "fold_t0":      float(t0),
        "fold_duration":float(duration),
        "bls_period":   float(period),
        "bls_depth":    float(bls_depth) if np.isfinite(bls_depth) else np.nan,
        "bls_duration": float(bls_dur)   if np.isfinite(bls_dur)   else np.nan,
        "bls_snr":      float(bls_snr),
        **{k: (float(v) if np.isfinite(float(v)) else np.nan)
           for k, v in eng.items()},
    }

    # Update view arrays
    new_gviews[ri] = gv
    new_lviews[ri] = lv

    # ── Propagate to augmented copies ────────────────────────────
    # The builder appended N_AUGMENTS copies immediately after each real row.
    # They all get the same label/scalars, but fresh noise on the views.
    aug_start = ri + 1
    for aug_offset in range(N_AUGMENTS):
        ai = aug_start + aug_offset
        if ai >= len(df):
            break
        if pd.isna(df.at[ai, "augmented"]) or int(df.at[ai, "target_id"]) != tic:
            break
        # Update scalars
        new_records[ai] = new_records[ri].copy()
        new_records[ai]["augmented"] = True
        # Re-noise the views with the same augmentation logic
        jitter    = int(rng.normal(0, PHASE_JITTER_STD * GLOBAL_BINS))
        g_aug     = np.roll(gv, jitter).copy().astype(np.float32)
        g_aug    += rng.normal(0, FLUX_NOISE_STD, size=g_aug.shape).astype(np.float32)
        l_aug     = (lv + rng.normal(0, FLUX_NOISE_STD,
                                     size=lv.shape).astype(np.float32))
        new_gviews[ai] = g_aug
        new_lviews[ai] = l_aug.astype(np.float32)
    
    if (pos + 1) % 50 == 0:
        print(f"  [{pos+1}/{len(real_idx)}]  {pos+1-skipped} updated, {skipped} skipped")

print(f"\nDone. {len(new_records)} rows updated ({skipped} skipped).")

# ── Write updated scalars back into df ───────────────────────────
update_df = pd.DataFrame.from_dict(new_records, orient="index")
for col in update_df.columns:
    if col in df.columns: 
        df.loc[update_df.index, col] = update_df[col].values
    else:
        df[col] = np.nan
        df.loc[update_df.index, col] = update_df[col].values

# Recompute scalar_features array from updated df
available_scalar = [c for c in SCALAR_COLS if c in df.columns and df[c].notna().any()]
new_scalar_feats = df[available_scalar].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)



Pre-processing raw light curves from superdf...
  1732 stars preprocessed.

Found 1732 real rows and 3464 augmented rows.
Running corrected BLS on 1732 real records...
  [50/1732]  50 updated, 0 skipped
  [100/1732]  100 updated, 0 skipped
  [150/1732]  150 updated, 0 skipped
  [200/1732]  200 updated, 0 skipped
  [250/1732]  250 updated, 0 skipped
  [300/1732]  300 updated, 0 skipped
  [350/1732]  350 updated, 0 skipped
  [400/1732]  400 updated, 0 skipped
  [450/1732]  450 updated, 0 skipped
  [500/1732]  500 updated, 0 skipped
  [550/1732]  550 updated, 0 skipped
  [600/1732]  600 updated, 0 skipped
  [650/1732]  650 updated, 0 skipped
  [700/1732]  700 updated, 0 skipped
  [750/1732]  750 updated, 0 skipped
  [800/1732]  800 updated, 0 skipped
  [850/1732]  850 updated, 0 skipped
  [900/1732]  900 updated, 0 skipped
  [950/1732]  950 updated, 0 skipped
  [1000/1732]  1000 updated, 0 skipped
  [1050/1732]  1050 updated, 0 skipped
  [1100/1732]  1100 updated, 0 skipped
  [1150/1732] 

In [2]:
# ── Overwrite saved files ─────────────────────────────────────────
import os
os.makedirs(OUT_DIR, exist_ok=True)

with open(f"{OUT_DIR}/final_tess_dataset.pkl", "wb") as fh:
    pickle.dump(df, fh)

np.save(f"{OUT_DIR}/global_views.npy",    new_gviews)
np.save(f"{OUT_DIR}/local_views.npy",     new_lviews)
np.save(f"{OUT_DIR}/labels.npy",          df["label"].values.astype(np.int64))
np.save(f"{OUT_DIR}/scalar_features.npy", new_scalar_feats)

# ── Reload in-memory variables to match saved state ──────────────
global_views  = new_gviews
local_views   = new_lviews
scalar_feats  = new_scalar_feats

# ── Summary ───────────────────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Files overwritten in {OUT_DIR}/")
print(f"  global_views  : {global_views.shape}")
print(f"  local_views   : {local_views.shape}")
print(f"  scalar_feats  : {scalar_feats.shape}")
print()
real_df = df[df["augmented"].isna()]
print(f"  BLS SNR (real records):")
print(f"    min    = {real_df['bls_snr'].min():.2f}")
print(f"    median = {real_df['bls_snr'].median():.2f}")
print(f"    max    = {real_df['bls_snr'].max():.2f}")
print()
print(f"  Label distribution (real + augmented):")
for lbl, name in {0:"Confirmed", 1:"Candidate", 2:"No signal"}.items():
    aug_bool = df["augmented"].notna()
    n_real = int(((df["label"] == lbl) & ~aug_bool).sum())
    n_aug  = int(((df["label"] == lbl) &  aug_bool).sum())
    print(f"    {lbl} {name:<12}: {n_real} real + {n_aug} aug = {n_real+n_aug}")
print(f"{'='*55}")


  Files overwritten in final_sector1_processed/
  global_views  : (5196, 2000)
  local_views   : (5196, 201)
  scalar_feats  : (5196, 23)

  BLS SNR (real records):
    min    = 2.97
    median = 6.59
    max    = 290.66

  Label distribution (real + augmented):
    0 Confirmed   : 424 real + 848 aug = 1272
    1 Candidate   : 370 real + 740 aug = 1110
    2 No signal   : 938 real + 1876 aug = 2814
